# XGBoost Baseline — LRG

Referencia para [../05_xgboost_optuna/xgb_optuna_lrg.ipynb](../05_xgboost_optuna/xgb_optuna_lrg.ipynb).


## 1. Setup

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, print_info

OBJECT_TYPE = "LRG"
print_info()

paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths["spectra_h5"].with_name(f"{OBJECT_TYPE}spectra_padded.h5")
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"HDF5:   {HDF5_PATH}")


PROJECT_ROOT: /home/thalita/spec_z_ml
DATA_BASE:    /home/thalita/spec_z_ml/data
  → raw:       /home/thalita/spec_z_ml/data/raw
  → processed: /home/thalita/spec_z_ml/data/processed
  → filtered:  /home/thalita/spec_z_ml/data/filtered
LOGS_DIR:     /home/thalita/spec_z_ml/logs
MODELS_DIR:   /home/thalita/spec_z_ml/models
RESULTS_DIR:  /home/thalita/spec_z_ml/results
Ambiente:     LOCAL

Objeto: LRG
HDF5:   /home/thalita/spec_z_ml/data/processed/LRG/LRGspectra_padded.h5


In [2]:
import numpy as np, pandas as pd

from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split, apply_split
from config import SPLITS_DIR
from src.models import train_xgboost
from src.evaluation import compute_redshift_metrics

SEED = 42
RESULTS_DIR = PROJECT_RESULTS / OBJECT_TYPE / "xgboost_baseline"
MODEL_DIR   = MODELS_DIR / OBJECT_TYPE / "xgboost_baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)


2026-05-27 14:46:15.800710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-27 14:46:15.884300: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-27 14:46:15.920512: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-27 14:46:15.932262: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-27 14:46:16.006550: I tensorflow/core/platform/cpu_feature_guar

## 2. Dados

In [3]:
X, y, _ = load_spectral_dataset(HDF5_PATH, seed=SEED)
X = normalize_spectra(X)
# Split canonico estratificado por z (mesmo de todos os modelos).
tr, va, te = make_or_load_split(OBJECT_TYPE, y, SPLITS_DIR)
X_train, X_val, X_test = apply_split(X, tr, va, te)
y_train, y_val, y_test = apply_split(y, tr, va, te)


## 3. Treinar

In [4]:
model = train_xgboost(X_train, y_train, X_val, y_val, seed=SEED)
y_pred = model.predict(X_test)
results = compute_redshift_metrics(y_test, y_pred)
print(f"RMSE: {results['rmse']:.4f}  R2: {results['r2']:.4f}  NMAD: {results['nmad']:.4f}")


RMSE: 0.0379  R2: 0.8127  NMAD: 0.0180


## 4. Salvar

In [ ]:
# Fix xgboost 3.x + sklearn 1.8: save_model() checa model._estimator_type,
# que o sklearn 1.8 removeu do RegressorMixin. Definimos manualmente.
model._estimator_type = "regressor"
model.save_model(MODEL_DIR / "xgboost_baseline.json")
np.savez(RESULTS_DIR / "predictions.npz", y_test=y_test, y_pred=y_pred, delta_z=results["delta_z"])
pd.DataFrame([{"model": "xgboost_baseline", "object": OBJECT_TYPE,
               **{k: v for k, v in results.items() if isinstance(v, (int, float))}}
            ]).to_csv(RESULTS_DIR / "metrics.csv", index=False)
